In [ ]:
import os
from glob import glob
from itertools import product

from PIL import Image, ImageDraw, ImageFont
from pillow_heif import HeifImagePlugin
from tqdm.contrib.itertools import product as tqdm_product

In [ ]:
# Image settings
FOLDER = "input"
RATIO = 3.0 / 4.0  # W / H
RATIO_TOLERANCE = 0.05
FILE_FORMATS = ("heic", "jpg", "png")

# Arrangement settings
# COLUMN_COUNT = 17
# ROW_COUNT = 22
# ZOOM_INSETS = (
#     # (TOP_LEFT_X, TOP_LEFT_Y, SIZE), INDEX
#     ((7, 9, 3), 364),
# )
COLUMN_COUNT = 38
ROW_COUNT = 10
ZOOM_INSETS = (
    # (TOP_LEFT_X, TOP_LEFT_Y, SIZE), INDEX
    ((17, 3, 4), 364),
)
ROW_MAJOR = True
# Image settings
DPI = 40
CANVAS_WIDTH_CM = 118.9
FONT_ALT = ImageFont.truetype("Arial.ttf", 40)
BORDER = ((0, 0, 0), 1)
BORDER_ALT = ((255, 0, 0), 5)

# Derived constants
IMAGE_WIDTH_PX = int(round(CANVAS_WIDTH_CM / 2.54 * DPI / COLUMN_COUNT))
IMAGE_HEIGHT_PX = int(round(IMAGE_WIDTH_PX / RATIO))
CANVAS_WIDTH_PX = IMAGE_WIDTH_PX * COLUMN_COUNT
CANVAS_HEIGHT_PX = IMAGE_HEIGHT_PX * ROW_COUNT
INSET_INDICES = [index for _, index in ZOOM_INSETS]
INSET_COORDINATES = [
    coordinates
    for (tlx, tly, size), _ in ZOOM_INSETS
    for coordinates in product(range(tlx, tlx + size), range(tly, tly + size))
]

canvas = Image.new(mode="RGB", size=(CANVAS_WIDTH_PX, CANVAS_HEIGHT_PX), color="white")
canvas_draw = ImageDraw.Draw(canvas)


def add_image(top_left_x, top_left_y, size, index):
    top_left_px = (top_left_x * IMAGE_WIDTH_PX, top_left_y * IMAGE_HEIGHT_PX)
    bottom_right_px = (
        top_left_px[0] + size * IMAGE_WIDTH_PX - 1,
        top_left_px[1] + size * IMAGE_HEIGHT_PX - 1,
    )
    for format in FILE_FORMATS:
        filename = os.path.join(FOLDER, f"{index}.{format}")
        try:
            image = Image.open(filename)
            ratio = image.width / image.height
            ratio_error = abs((ratio - RATIO) / RATIO)
            if ratio_error > RATIO_TOLERANCE:
                print(f"Incorrect ratio for image {index} ({ratio:.2f} vs. {RATIO:.2f})")
            image_resized = image.resize(
                (size * IMAGE_WIDTH_PX, size * IMAGE_HEIGHT_PX), resample=Image.LANCZOS
            )
            canvas.paste(image_resized, box=top_left_px)
            canvas_draw.rectangle(
                (top_left_px, bottom_right_px), outline=BORDER[0], width=BORDER[1]
            )
            image.close()
            break
        except FileNotFoundError:
            if format == FILE_FORMATS[-1]:
                center_px = (
                    top_left_px[0] + size * IMAGE_WIDTH_PX // 2,
                    top_left_px[1] + size * IMAGE_HEIGHT_PX // 2,
                )
                canvas_draw.text(
                    center_px,
                    f"{index}",
                    anchor="ms",
                    font=FONT_ALT,
                    fill=(255, 0, 0),
                )
                canvas_draw.rectangle(
                    (top_left_px, bottom_right_px), outline=BORDER_ALT[0], width=BORDER_ALT[1]
                )


if ROW_MAJOR:
    coordinates_iterator = tqdm_product(range(ROW_COUNT), range(COLUMN_COUNT))
else:
    coordinates_iterator = tqdm_product(range(COLUMN_COUNT), range(ROW_COUNT))
print("Processing images ...")
current_index = 0
for coordinates in coordinates_iterator:
    if ROW_MAJOR:
        row, column = coordinates
    else:
        column, row = coordinates
    if (column, row) in INSET_COORDINATES:
        continue

    current_index += 1
    while current_index in INSET_INDICES:
        current_index += 1

    add_image(column, row, 1, current_index)

for (top_left_x, top_left_y, size), index in ZOOM_INSETS:
    add_image(top_left_x, top_left_y, size, index)


print("Saving file ...")
canvas.save("output/merge.png")